# Desigualdad estructural y calidad educativa en Ecuador

### Notebook exploratorio y metodológico (versión por relaciones)

**Pregunta de investigación**

> ¿Cómo se transmiten las desigualdades de origen socioeconómico a través de la calidad
> educativa hacia las oportunidades en Ecuador, y cómo se compara su eficiencia de recursos
> educativos con la de otros países?

Este notebook organiza la exploración alrededor de tres relaciones, siguiendo un enfoque
**descriptivo y correlacional** (asociaciones y patrones, no causalidad):

- **R1. Origen socioeconómico → Calidad educativa.** ¿El nivel socioeconómico de la familia y
  la educación de los padres se asocian con el aprendizaje? Fuentes: ERCE 2019 (nivel
  estudiante, comparable entre países) e INEVAL Ser Estudiante (interno a Ecuador).
- **R3. Recursos → Resultados.** ¿La inversión y el ratio alumno-docente se asocian con el
  aprendizaje a nivel país? Fuentes: OWID (LAYS, gasto, ratio), UNESCO UIS, Banco Mundial,
  Cuentas Satélite.
- **R2. Educación → Oportunidades.** ¿El nivel educativo se asocia con ingreso y empleo?
  Fuentes: SEDLAC (regional) y ENEMDU microdatos (pendiente de descarga).

Es una versión de validación, no la final. El objetivo es comprobar, con los datos ya en mano,
que cada relación se puede sostener con evidencia, dejando explícito qué no se puede afirmar.

Nota metodológica importante: se adoptan los datasets de la investigación complementaria, pero
no su aparato causal (Mincer-Heckman, variables instrumentales, modelos jerárquicos). El diseño
es transversal y observacional.

## 0. Configuración y utilidades

Se centralizan aquí las funciones de lectura para no repetir lógica. Cada fuente tiene su
particularidad: INEVAL usa `;` y coma decimal; ERCE viene en `.sav` (SPSS) con pesos y valores
plausibles; OWID y SEDLAC son descargas locales; UIS y Banco Mundial son APIs.

In [1]:
import os, io, re, json, urllib.request
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import pyreadstat

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

DATA = Path("data")
ERCE = DATA / "erce-2019" / "sav"
assert DATA.exists(), "Ejecutar desde la raiz del proyecto (donde esta la carpeta data/)."
print("Directorio:", os.getcwd())
print("Fuentes en data/:", sorted(p.name for p in DATA.iterdir() if p.is_dir()))

Directorio: /sessions/eager-fervent-dirac/mnt/concurso-visualizador
Fuentes en data/: ['cuentas-satelites-servicios-educacion', 'encuesta-nacional-empleo-desempleo-subempleo', 'erce-2019', 'ineval-ser-estudiante', 'openstreetmap-ecu', 'owid', 'pisa-data-files', 'sedlac', 'senecyt', 'ser-bachiller', 'ser-profesional']


In [2]:
# --- INEVAL (evaluaciones Ser): separador ; y coma decimal; sentinelas de ausencia ---
NA_INEVAL = ["", " ", "999999", "*999999", "999999.0"]
def leer_ser(path, nrows=None, usecols=None):
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, sep=";", decimal=",", na_values=NA_INEVAL,
                               encoding=enc, low_memory=False, nrows=nrows, usecols=usecols)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("enc", b"", 0, 1, "ni utf-8 ni latin-1")

# --- media ponderada robusta (ignora nulos) ---
def wmean(x, w):
    ok = x.notna() & w.notna()
    return np.average(x[ok], weights=w[ok]) if ok.sum() else np.nan

# --- acceso generico a APIs JSON ---
def api_get(url):
    req = urllib.request.Request(url, headers={"Accept": "application/json",
                                               "User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=40) as r:
        return json.load(r)

# --- Banco Mundial: respuesta [metadatos, registros] ---
def wb_get(indicador, paises="ECU", date="2010:2023", per_page=500):
    url = (f"https://api.worldbank.org/v2/country/{paises}/indicator/{indicador}"
           f"?format=json&date={date}&per_page={per_page}")
    data = api_get(url)
    if not isinstance(data, list) or len(data) < 2 or data[1] is None:
        return pd.DataFrame()
    return pd.DataFrame([{"pais": x["countryiso3code"], "anio": int(x["date"]),
                          "valor": x["value"]} for x in data[1]])

# --- UNESCO UIS: separador de paises es COMA (no +) ---
UIS = "https://api.uis.unesco.org/api/public"
def uis_get(indicador, geo="ECU", start=2010, end=2024):
    geo = ",".join(geo) if isinstance(geo, (list, tuple)) else geo
    data = api_get(f"{UIS}/data/indicators?indicator={indicador}&geoUnit={geo}"
                   f"&start={start}&end={end}&indicatorMetadata=false")
    return pd.DataFrame(data.get("records", []))

print("Utilidades cargadas.")

Utilidades cargadas.


## 1. Inventario de datos disponibles

Tras depurar el repositorio (se retiraron el Censo 2022 y la ECV 2013-2014 por estar fuera del
alcance descriptivo de las tres relaciones, y las copias redundantes de ERCE), este es el estado
actual.

In [3]:
def peso_mb(p):
    t = sum(os.path.getsize(os.path.join(r, f))
            for r, _, fs in os.walk(p) for f in fs if os.path.exists(os.path.join(r, f)))
    return round(t / 1e6, 1)

inv = pd.DataFrame([(p.name, peso_mb(p)) for p in sorted(DATA.iterdir()) if p.is_dir()],
                   columns=["fuente", "peso_MB"]).sort_values("peso_MB", ascending=False)
inv.reset_index(drop=True)

,fuente,peso_MB
0,erce-2019,426.6
1,openstreetmap-ecu,119.3
2,pisa-data-files,80.0
3,ineval-ser-estudiante,53.4
4,ser-bachiller,49.1
5,senecyt,13.4
6,cuentas-satelites-servicios-educacion,7.3
7,sedlac,6.4
8,ser-profesional,0.9
9,encuesta-nacional-empleo-desempleo-subempleo,0.7


Asignación de cada fuente a las relaciones:

- **R1:** ERCE 2019 (estudiante + familia + docente), INEVAL Ser Estudiante, Ser Bachiller.
- **R3:** OWID (LAYS, gasto, ratio), Cuentas Satélite (gasto Ecuador), UIS y Banco Mundial (API).
- **R2:** SEDLAC (regional); ENEMDU microdatos pendiente de descarga.
- Contexto/descartadas: PISA (Ecuador no participa), OSM (solo mapas), SENESCYT y ENEMDU
  tabulados 2017 (contexto agregado).

## 2. R1 — Origen socioeconómico → Calidad educativa

Es la relación mejor sostenida del proyecto, porque se puede medir a **nivel de estudiante** y,
con ERCE, también comparar con la región usando la misma prueba.

### 2.1 Señal interna en Ecuador (INEVAL Ser Estudiante)

INEVAL trae, por estudiante, la nota global (`inev`) y un índice socioeconómico (`isec`) con su
quintil. Primera pregunta concreta: ¿la nota sube con el nivel socioeconómico?

In [4]:
ser = leer_ser(DATA / "ineval-ser-estudiante" / "2023-2024" / "data.csv")
print("Ser Estudiante 2023-2024:", ser.shape)

# Nota global por quintil socioeconomico (ponderada por factor de expansion fex_inev)
g = ser.dropna(subset=["quintil", "inev"])
tab_q = g.groupby("quintil").apply(
    lambda d: pd.Series({"nota_media": wmean(d["inev"], d["fex_inev"]),
                         "n": len(d)}), include_groups=False)
print("\nNota global por quintil socioeconomico (ponderada):")
print(tab_q.round(1).to_string())

Ser Estudiante 2023-2024: (50545, 78)



Nota global por quintil socioeconomico (ponderada):
         nota_media       n
quintil                    
1.0           681.0  7542.0
2.0           684.5  7542.0
3.0           688.1  7542.0
4.0           695.0  7542.0
5.0           701.9  7542.0


In [5]:
# La misma nota por area (rural/urbano) y por financiamiento (publico/privado)
mapa_area = {1: "Rural", 2: "Urbano"}
mapa_fin = {1: "Publico", 2: "Privado", 3: "Fiscomisional"}
g2 = ser.dropna(subset=["inev"])
print("Nota global por area:")
print(g2.assign(area=g2["tp_area"].map(mapa_area))
        .groupby("area").apply(lambda d: wmean(d["inev"], d["fex_inev"]), include_groups=False)
        .round(1).to_string())
print("\nNota global por financiamiento:")
print(g2.assign(fin=g2["financiamiento"].map(mapa_fin))
        .groupby("fin").apply(lambda d: wmean(d["inev"], d["fex_inev"]), include_groups=False)
        .round(1).to_string())

Nota global por area:
area
Rural     684.6
Urbano    688.4

Nota global por financiamiento:
fin
Fiscomisional    690.8
Privado          703.7
Publico          683.6


La nota sube de forma monótona con el quintil, es mayor en lo urbano que en lo rural, y mayor
en privado que en público. Son tres cortes de la misma desigualdad estructural. INEVAL, sin
embargo, no trae la educación de los padres de forma interpretable, y no compara con otros
países. Ahí entra ERCE.

### 2.2 ERCE 2019: puntaje de Ecuador frente a la región

ERCE evalúa 6.° de primaria en 16 países con la misma prueba. El desempeño se entrega como cinco
**valores plausibles** por dominio (Matemática `MAT`, Lenguaje `LAN`, Ciencias `SCI`); el puntaje
del estudiante es su promedio, y las medias de país se ponderan con `WT`. Se carga una sola vez y
se reutiliza.

In [6]:
dom = ["MAT", "LAN", "SCI"]
pv_cols = [f"{d}_{i}" for d in dom for i in range(1, 6)]
qa6, _ = pyreadstat.read_sav(str(ERCE / "ERCE_2019_QA6.sav"),
                             usecols=["IDSTUD", "IDCNTRY", "COUNTRY", "WT"] + pv_cols)
for d in dom:
    qa6[d] = qa6[[f"{d}_{i}" for i in range(1, 6)]].mean(axis=1)
print("QA6 (alumno 6.° grado):", qa6.shape, "| paises:", qa6["COUNTRY"].nunique())

punt = pd.DataFrame({d: qa6.groupby("COUNTRY").apply(
    lambda g: wmean(g[d], g["WT"]), include_groups=False) for d in dom})
print("\nPuntaje medio ponderado por pais (ERCE 2019, 6.° grado):")
print(punt.round(0).sort_values("MAT", ascending=False).to_string())

QA6 (alumno 6.° grado): (80827, 22) | paises: 16

Puntaje medio ponderado por pais (ERCE 2019, 6.° grado):
           MAT    LAN    SCI
COUNTRY                     
PER      759.0  741.0  723.0
URY      759.0  734.0  731.0
MEX      758.0  726.0  726.0
BRA      733.0  734.0  718.0
CRI      726.0  757.0  758.0
ECU      720.0  684.0  720.0
COL      707.0  719.0  711.0
ARG      690.0  698.0  682.0
CUB      689.0  738.0  779.0
HND      682.0  661.0  674.0
SLV      676.0  699.0  705.0
NIC      663.0  654.0  669.0
GTM      657.0  645.0  661.0
PRY      647.0  657.0  657.0
PAN      645.0  652.0  672.0
DOM      636.0  644.0  649.0


In [7]:
media_reg = punt.drop(index="ECU").mean()
print("Ecuador vs promedio regional (sin Ecuador):")
comp = pd.DataFrame({"Ecuador": punt.loc["ECU"], "Media_region": media_reg,
                     "Brecha": punt.loc["ECU"] - media_reg}).round(1)
print(comp.to_string())

Ecuador vs promedio regional (sin Ecuador):
     Ecuador  Media_region  Brecha
MAT    719.8         695.2    24.6
LAN    684.3         697.3   -13.1
SCI    720.1         701.1    19.0


Ecuador queda en la zona media: por encima del promedio regional en Matemática y Ciencias,
pero **por debajo en Lenguaje**. Es un primer hallazgo sustantivo y honesto: la debilidad relativa
ecuatoriana está en lectura, no en matemática.

### 2.3 ERCE: el gradiente socioeconómico a nivel estudiante

Aquí está lo que ninguna otra fuente local permitía. El cuestionario de familia trae un índice
socioeconómico ya calculado (`ISECF`), la educación de la madre (`FFIT11`) y los libros en casa
(`FFIT19`). Se une al puntaje por la clave compuesta `IDSTUD + IDCNTRY` (el ID de estudiante se
repite entre países, así que la unión debe incluir el país).

In [8]:
qf6, meta_qf = pyreadstat.read_sav(str(ERCE / "ERCE_2019_QF6.sav"),
                                   usecols=["IDSTUD", "IDCNTRY", "ISECF", "FFIT11", "FFIT19"])
erce = qa6.merge(qf6, on=["IDSTUD", "IDCNTRY"], how="inner")
print("Union puntaje + familia:", erce.shape, "(debe rondar las 80.827 filas)")

ecu = erce[erce["COUNTRY"] == "ECU"].copy()
ecu["q_isec"] = pd.qcut(ecu["ISECF"], 5, labels=[1, 2, 3, 4, 5])
grad = ecu.groupby("q_isec", observed=True).apply(
    lambda g: wmean(g["MAT"], g["WT"]), include_groups=False)
print("\nEcuador 6.°: Matematica por quintil socioeconomico familiar (ISECF):")
print(grad.round(0).to_string())
print("Brecha Q5 - Q1:", round(grad.loc[5] - grad.loc[1], 0), "puntos")

Union puntaje + familia: (80827, 25) (debe rondar las 80.827 filas)

Ecuador 6.°: Matematica por quintil socioeconomico familiar (ISECF):
q_isec
1    686.0
2    699.0
3    717.0
4    739.0
5    770.0
Brecha Q5 - Q1: 84.0 puntos


In [9]:
# Mismo gradiente segun educacion de la madre (FFIT11), agrupando niveles
lab_madre = meta_qf.variable_value_labels.get("FFIT11", {})
ecu["educ_madre"] = ecu["FFIT11"].map(lab_madre)
nivel = (ecu.dropna(subset=["FFIT11", "MAT"])
            .groupby("FFIT11")
            .apply(lambda g: pd.Series({"nivel": lab_madre.get(g.name, g.name),
                                        "MAT": wmean(g["MAT"], g["WT"]), "n": len(g)}),
                   include_groups=False))
print("Matematica segun nivel educativo de la madre (Ecuador, 6.°):")
print(nivel[["nivel", "MAT", "n"]].to_string(index=False))

Matematica segun nivel educativo de la madre (Ecuador, 6.°):
                   nivel        MAT    n
       No tiene estudios 669.881658  451
<CINE-P 1-2> incompleta. 708.210889 1310
  <CINE-P 1-2> completa. 705.788353  665
  <CINE-P 3> incompleta. 699.019278  423
    <CINE-P 3> completa. 728.763786 1564
  <CINE-P 4> incompleta. 729.480822  181
    <CINE-P 4> completa. 743.497523  297
<CINE-P 5-6> incompleta. 767.871086  271
  <CINE-P 5-6> completa. 779.138016  677
<CINE-P 7-8> incompleto. 743.102503   27
  <CINE-P 7-8> completo. 807.372573  110
         No sé/No aplica 683.413107  233


El gradiente es nítido: ~84 puntos de diferencia entre el quintil socioeconómico más bajo y
el más alto (del orden de dos años de escolaridad en la escala ERCE), y la nota crece con la
educación de la madre. Esto sostiene directamente R1.

### 2.4 ¿La desigualdad de Ecuador es mayor o menor que la de la región?

Se calcula la misma brecha Q5-Q1 dentro de cada país. Permite ubicar a Ecuador no solo por su
nivel, sino por cuán desigual es su sistema.

In [10]:
def brecha_pais(df, dominio="MAT"):
    out = []
    for c, g in df.groupby("COUNTRY"):
        gg = g.dropna(subset=["ISECF", dominio])
        if len(gg) < 200:
            continue
        gg = gg.assign(q=pd.qcut(gg["ISECF"], 5, labels=[1, 2, 3, 4, 5]))
        mq = gg.groupby("q", observed=True).apply(
            lambda h: wmean(h[dominio], h["WT"]), include_groups=False)
        out.append((c, round(mq.loc[5] - mq.loc[1], 0)))
    return pd.DataFrame(out, columns=["pais", "brecha_Q5_Q1"]).sort_values("brecha_Q5_Q1", ascending=False)

br = brecha_pais(erce, "MAT")
print("Brecha socioeconomica Q5-Q1 en Matematica, por pais (ERCE 6.°):")
print(br.to_string(index=False))
print("\nPosicion de Ecuador:", br.reset_index(drop=True).query("pais=='ECU'").index[0] + 1,
      "de", len(br))

Brecha socioeconomica Q5-Q1 en Matematica, por pais (ERCE 6.°):
pais  brecha_Q5_Q1
 BRA         140.0
 URY         126.0
 PAN         118.0
 GTM         114.0
 COL         112.0
 PER         109.0
 CRI         107.0
 SLV          94.0
 ARG          92.0
 ECU          84.0
 MEX          84.0
 PRY          80.0
 DOM          75.0
 HND          38.0
 CUB          35.0
 NIC          31.0

Posicion de Ecuador: 10 de 16


## 3. R3 — Recursos → Resultados

Pregunta: ¿más gasto o menos alumnos por docente se asocian con más aprendizaje? Aquí la unidad
es el **país**, no el estudiante. Es una mirada ecológica y descriptiva: sirve para situar a
Ecuador en la nube internacional, no para inferir causalidad (pocos casos, sin control de
confusores).

### 3.1 OWID: aprendizaje (LAYS) frente a gasto y a ratio alumno-docente

LAYS = años de escolaridad ajustados por aprendizaje (funde cantidad y calidad). Se usan las
descargas locales de OWID.

In [11]:
def owid_latest(fname, valcol, nuevo):
    df = pd.read_csv(DATA / "owid" / fname)
    df = df.rename(columns={valcol: nuevo}).dropna(subset=[nuevo])
    return df.sort_values("Year").groupby("Code").tail(1).set_index("Code")[nuevo]

lays  = owid_latest("years-of-schooling.csv", "Both genders", "LAYS")
gasto = owid_latest("education-spending.csv", "Total across all levels of education", "gasto_pib")
ratio = owid_latest("pupil-teacher-ratio-for-primary-education-by-country.csv",
                    "Pupil-qualified teacher ratio in primary education", "ratio_ad")

panel = pd.concat([lays, gasto, ratio], axis=1).dropna(subset=["LAYS"])
print("Panel pais (ultimo dato disponible por indicador):", panel.shape)
print("\nCorrelaciones con LAYS (Pearson):")
print(panel.corr(numeric_only=True)["LAYS"].round(2).to_string())
print("\nEcuador en el panel:")
print(panel.loc[["ECU", "CHL", "PER", "COL", "KOR", "FIN"]].round(2).to_string())

Panel pais (ultimo dato disponible por indicador): (174, 3)

Correlaciones con LAYS (Pearson):
LAYS         1.00
gasto_pib    0.13
ratio_ad    -0.56

Ecuador en el panel:
       LAYS  gasto_pib  ratio_ad
Code                            
ECU    8.70       3.69     23.25
CHL    9.41       4.91     18.18
PER    8.63       4.36     21.38
COL    8.62       5.26     23.46
KOR   11.68       5.41     14.48
FIN   11.74       6.38       NaN


La correlación esperada aparece (a más gasto, más LAYS; a más alumnos por docente, menos
LAYS), pero es una asociación a nivel país: no implica causalidad y está confundida por el
ingreso del país. Sirve para describir dónde se ubica Ecuador, no para afirmar que gastar más
*causa* aprender más (esa es la "paradoja de Hanushek" que el análisis no puede resolver con
estos datos).

### 3.2 Indicadores de recursos por API (Banco Mundial y UNESCO UIS)

Para los indicadores que no están en OWID se usan las APIs. Siguiendo la regla del proyecto, se
parte de su documentación. La de la UIS es una página Swagger, pero expone un esquema OpenAPI
legible que enumera endpoints y parámetros.

In [12]:
schema = api_get("https://api.uis.unesco.org/api/public/openapi/schema.json")
print("UIS API:", schema["info"]["title"], schema["info"]["version"])
for path, methods in schema["paths"].items():
    for m, op in methods.items():
        ps = [p["name"] for p in op.get("parameters", []) if p["name"] != "Accept-Encoding"]
        print(f"  {m.upper()} {path}  params={ps}")

UIS API: UIS Data API 1.0.2
  GET /api/public/data/indicators  params=['indicator', 'geoUnit', 'geoUnitType', 'start', 'end', 'footnotes', 'indicatorMetadata', 'version']
  GET /api/public/data/indicators/export  params=['indicator', 'geoUnit', 'geoUnitType', 'start', 'end', 'footnotes', 'indicatorMetadata', 'version', 'format']
  GET /api/public/definitions/geounits  params=['version']
  GET /api/public/definitions/geounits/export  params=['version', 'format']
  GET /api/public/definitions/indicators  params=['version', 'glossaryTerms', 'disaggregations']
  GET /api/public/definitions/indicators/export  params=['version', 'glossaryTerms', 'disaggregations', 'format']
  GET /api/public/versions/default  params=[]
  GET /api/public/versions  params=[]


In [13]:
# Banco Mundial: gasto educativo (%PIB) e internet, para Ecuador y comparadores
print("Banco Mundial - gasto publico en educacion (% PIB), ultimo dato:")
wb = wb_get("SE.XPD.TOTL.GD.ZS", "ECU;CHL;PER;COL;KOR;FIN", "2015:2022").dropna(subset=["valor"])
print(wb.sort_values("anio").groupby("pais").tail(1).sort_values("valor", ascending=False)
        [["pais", "anio", "valor"]].to_string(index=False))

print("\nUNESCO UIS - ratio alumno/docente calificado en primaria (PTRHC.1.QUALIFIED):")
u = uis_get("PTRHC.1.QUALIFIED", ["ECU", "CHL", "PER", "COL"], 2015, 2024).dropna(subset=["value"])
print(u.sort_values("year").groupby("geoUnit").tail(1)[["geoUnit", "year", "value"]].to_string(index=False))

Banco Mundial - gasto publico en educacion (% PIB), ultimo dato:


pais  anio   valor
 FIN  2022 6.37578
 KOR  2022 5.40910
 COL  2020 5.26166
 CHL  2022 4.91201
 PER  2022 3.83975
 ECU  2022 3.61302

UNESCO UIS - ratio alumno/docente calificado en primaria (PTRHC.1.QUALIFIED):


geoUnit  year     value
    CHL  2016 18.182980
    COL  2022 23.458179
    PER  2023 21.377131
    ECU  2024 23.253197


### 3.3 Cuentas Satélite de Educación (gasto de Ecuador en el tiempo)

Aporta el detalle nacional del gasto educativo 2007-2024, pero viene como cuadros de reporte
(con encabezados y codificación latina), no como tabla limpia. Para la serie comparable conviene
apoyarse en OWID/Banco Mundial; las Cuentas Satélite quedan como fuente de profundización
nacional que requiere limpieza. Se muestra su estructura cruda para dejar constancia.

In [14]:
cse = (DATA / "cuentas-satelites-servicios-educacion" / "data" /
       "6_Indicadores_FyE_CSE_2007-24" / "1.1_GNE_PIB.csv")
crudo = pd.read_csv(cse, sep=",", encoding="latin-1", header=None, nrows=8)
print("Estructura cruda (requiere parsing antes de usar):")
print(crudo.iloc[:6, :6].to_string())

Estructura cruda (requiere parsing antes de usar):
                                                                                                    0   1   2   3   4   5
0                                                                                              Índice NaN NaN NaN NaN NaN
1                                                                                                 NaN NaN NaN NaN NaN NaN
2                                                                                       CUADRO N° 1.1 NaN NaN NaN NaN NaN
3  Gasto Nacional en Educación según sector público y privado respecto del PIB\r\r\nPeríodo 2007-2024 NaN NaN NaN NaN NaN
4                                                                                                 NaN NaN NaN NaN NaN NaN
5                                                                                    Miles de dólares NaN NaN NaN NaN NaN


## 4. R2 — Educación → Oportunidades (andamiaje)

Aquí hay que ser honestos con el alcance: ningún dato enlaza la **nota** de un estudiante con su
salario futuro. Lo que se puede medir es el retorno del **nivel educativo alcanzado** sobre
ingreso y empleo. Es una relación de atributos y retornos, no de "calidad → salario".

### 4.1 SEDLAC: retornos y educación en la región (descarga local)

SEDLAC armoniza encuestas de hogares de América Latina y publica estadígrafos ya calculados
(años de educación, salarios, Gini educativo), útiles para el benchmarking regional sin procesar
microdatos de cada país.

In [15]:
sedlac_dir = DATA / "sedlac"
print("Archivos SEDLAC disponibles:")
for f in sorted(sedlac_dir.glob("*.xlsx")):
    xl = pd.ExcelFile(f)
    print(f"  {f.name}: hojas {xl.sheet_names[:6]}")

Archivos SEDLAC disponibles:
  2025_Act1_employment_LAC.xlsx: hojas ['Index', 'labor force', 'employment', 'unemployment', 'duration', 'change']
  2025_Act1_enrollment_LAC.xlsx: hojas ['index', 'gender', 'income', 'area', 'primary', 'secondary']
  2025_Act1_literacy_LAC.xlsx: hojas ['index', 'age_gender', 'income', 'area']


  2025_Act1_wages_hours_LAC.xlsx: hojas ['Index', 'wage_1', 'wage_2', 'wage_3', 'wage_4', 'hours_1']
  2025_Act1_years_edu_LAC.xlsx: hojas ['index', 'structure', 'years', 'age_gender', 'income', 'income_age']


### 4.2 ENEMDU microdatos — lector listo, pendiente de descarga

Los microdatos de persona y hogar de la ENEMDU (catálogo 1270 del ANDA-INEC, diciembre 2023:
`enemdu_persona_2023_12` y `enemdu_vivienda_hogar_2023_12`) permiten describir la relación entre
nivel educativo e ingreso/empleo. Se deja la función de lectura preparada; el notebook se ejecuta
sin error aunque los archivos no estén todavía. **No se generan datos sintéticos.**

Al recibirlos se validará: variables de diseño muestral (`upm`, `fexp`, `estrato`) obligatorias,
nivel de instrucción, ingreso laboral y condición de actividad. Limitación: el rediseño muestral
2020-2021 rompe la comparabilidad histórica.

In [16]:
ENEMDU_DIR = DATA / "enemdu-microdatos"
def cargar_enemdu_persona():
    objetivo = list(ENEMDU_DIR.glob("*persona*")) if ENEMDU_DIR.exists() else []
    if not objetivo:
        print("ENEMDU pendiente de descarga. Esperado en:", ENEMDU_DIR)
        print("Archivos: enemdu_persona_2023_12 y enemdu_vivienda_hogar_2023_12 (ANDA-INEC cat. 1270).")
        return None
    f = objetivo[0]
    df = pd.read_csv(f, sep=";", decimal=",", low_memory=False) if f.suffix.lower() == ".csv" \
        else pyreadstat.read_sav(str(f))[0]
    print("ENEMDU cargada:", f.name, df.shape)
    return df

enemdu = cargar_enemdu_persona()

ENEMDU pendiente de descarga. Esperado en: data/enemdu-microdatos
Archivos: enemdu_persona_2023_12 y enemdu_vivienda_hogar_2023_12 (ANDA-INEC cat. 1270).


## 5. Validación metodológica

### Qué se puede sostener
1. **R1, fuerte.** La nota se asocia con el nivel socioeconómico, dentro de Ecuador (INEVAL,
   quintil/área/financiamiento) y a nivel estudiante con educación de los padres (ERCE). La
   comparación internacional sitúa a Ecuador y describe su debilidad en lectura.
2. **R3, descriptiva.** Asociación país entre recursos (gasto, ratio) y aprendizaje (LAYS); útil
   para ubicar a Ecuador, no para causalidad.
3. **R2, parcial.** Retornos del nivel educativo (no de la calidad) sobre ingreso/empleo, con
   SEDLAC y, al descargarla, ENEMDU.

### Qué NO se puede sostener
- Causalidad (todo es transversal y observacional).
- Vincular la nota individual con resultados laborales futuros (ninguna fuente lo permite).
- Ubicar a Ecuador en PISA/TIMSS/PIRLS (no participa).
- Trayectorias individuales en el tiempo (no hay panel).

### Cuidados técnicos obligatorios
- **Valores plausibles** (ERCE): usar el promedio de los cinco, no uno solo; para inferencia
  estricta, promediar resultados sobre los cinco. **Pesos** `WT` (ERCE) y `fex_*` (INEVAL)
  siempre. ENEMDU exige `upm`/`estrato`/`fexp`.
- **Nivel ecológico** en R3: no trasladar correlaciones país a conclusiones sobre individuos.
- **Desfase temporal**: ERCE 2019, INEVAL 2022-2025, ENEMDU 2023, indicadores ~2020-2022. No
  cruzar años distintos como si fueran el mismo momento.

## 6. Conclusión y siguientes pasos

Con los datos ya validados, las tres relaciones tienen sustento en su versión descriptiva. R1 es
el corazón del proyecto y está completamente operativo (gradiente socioeconómico medible a nivel
estudiante, comparable con la región). R3 funciona como descripción ecológica del lugar de
Ecuador frente al mundo. R2 queda lista en cuanto se descargue ENEMDU.

**Pendiente único para cerrar el alcance:** descargar los microdatos de ENEMDU 2023 (persona y
hogar) y colocarlos en `data/enemdu-microdatos/`. Con eso, las tres relaciones quedan completas y
el siguiente paso es seleccionar las visualizaciones (una por pregunta) para el entregable.